# 🚀 Exercise 5: FastText Embeddings & Comparison with Word2Vec

## 🛠️ Step 1: Import Libraries and Download NLTK Resources

In [1]:
import re
import string
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec, FastText

# Download required resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("Libraries and resources loaded successfully!")

Libraries and resources loaded successfully!


## 📝 Step 2: Define and Preprocess the Extended Corpus

In [2]:
# The combined corpus of 30 product/tech review sentences
corpus = [
    # Core 20 sentences
    "The smartphone has an incredible display with vivid colors and sharp resolution.",
    "Battery life is outstanding and lasts more than two days on a single charge.",
    "The camera produces stunning photographs even in low light conditions.",
    "Customer service was very helpful and resolved my issue within minutes.",
    "The laptop keyboard feels comfortable and typing experience is smooth.",
    "Delivery was fast and the packaging was secure preventing any damage.",
    "The sound quality of these headphones is rich, deep, and immersive.",
    "Screen resolution on this monitor is crystal clear and eye friendly.",
    "The gaming performance of this GPU is exceptional at high settings.",
    "This smartwatch tracks fitness data accurately including heart rate and sleep.",
    "Software updates are frequent and the operating system runs smoothly.",
    "The build quality feels premium with a solid aluminum chassis design.",
    "Connectivity options include USB-C, HDMI, and fast wireless Bluetooth.",
    "The streaming service offers thousands of movies and series in HD quality.",
    "Setup was straightforward and the user manual explains every step clearly.",
    "Price is reasonable compared to competitors offering similar specifications.",
    "The touchscreen response is highly accurate with minimal input latency.",
    "Voice assistant integration works seamlessly with smart home devices.",
    "Return policy is flexible and refunds are processed within three days.",
    "Overall this product exceeded my expectations and I highly recommend it.",
    # Extra 10 sentiment-focused sentences
    "The product is absolutely amazing and I love the excellent performance.",
    "Terrible experience, the device is horrible and completely broken.",
    "This is the best purchase I have ever made, truly outstanding quality.",
    "Worst product ever, totally disappointed and very frustrated with it.",
    "Fantastic build, the premium design looks beautiful and feels perfect.",
    "Awful customer support, the response was rude and completely unhelpful.",
    "The excellent display is bright, vivid, and delivers superb clarity.",
    "Very poor quality, the cheap materials feel fragile and look ugly.",
    "I am very happy and satisfied with this wonderful and brilliant device.",
    "Dreadful performance, the sluggish system crashes and freezes constantly."
]

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens 
              if w not in stop_words and len(w) > 1]
    return tokens

tokenized_corpus = [preprocess(sent) for sent in corpus]
total_tokens = sum(len(s) for s in tokenized_corpus)

print(f"Total sentences in corpus : {len(tokenized_corpus)}")
print(f"Total processed tokens    : {total_tokens}")

Total sentences in corpus : 30
Total processed tokens    : 213



For a fair comparison, we train both models with identical hyperparameters:
- `vector_size=100`
- `window=5`
- `min_count=1`
- `sg=1` (Skip-gram architecture)
- `epochs=200`
- `seed=42`
- `workers=1` (For complete reproducibility)

In [3]:
w2v_model = Word2Vec(
    tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=200,
    seed=42,
    workers=1
)

# Train FastText model
ft_model = FastText(
    tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=200,
    seed=42,
    workers=1
)

w2v_vocab = list(w2v_model.wv.index_to_key)
ft_vocab = list(ft_model.wv.index_to_key)

print(f"Word2Vec Vocabulary Size : {len(w2v_vocab)}")
print(f"FastText Vocabulary Size : {len(ft_vocab)}")
print(f"Vocabularies are identical: {set(w2v_vocab) == set(ft_vocab)}")

Word2Vec Vocabulary Size : 180
FastText Vocabulary Size : 180
Vocabularies are identical: True


##   Comparison Experiments

In [4]:
oov_words = ['smartphones', 'cameras', 'unfriendly', 'chargers', 'beautifully']

print(f"{'OOV Word':<15} | {'Word2Vec Status':<25} | {'FastText Status':<25}")
print("-" * 72)

for word in oov_words:
    try:
        w2v_vec = w2v_model.wv[word]
        w2v_status = "SUCCESS"
    except KeyError:
        w2v_status = "FAILED (KeyError)"
        
    # Test FastText
    try:
        ft_vec = ft_model.wv[word]
        ft_status = f"SUCCESS (Norm: {np.linalg.norm(ft_vec):.4f})"
    except KeyError:
        ft_status = "FAILED"
        
    print(f"{word:<15} | {w2v_status:<25} | {ft_status:<25}")

OOV Word        | Word2Vec Status           | FastText Status          
------------------------------------------------------------------------
smartphones     | FAILED (KeyError)         | SUCCESS (Norm: 1.0793)   
cameras         | FAILED (KeyError)         | SUCCESS (Norm: 1.0370)   
unfriendly      | FAILED (KeyError)         | SUCCESS (Norm: 0.9360)   
chargers        | FAILED (KeyError)         | SUCCESS (Norm: 0.8812)   
beautifully     | FAILED (KeyError)         | SUCCESS (Norm: 0.8862)   


Since Word2Vec cannot represent them, we can't find similar words for them. But FastText can!

In [5]:
print("FastText - Most Similar In-Vocabulary Words for OOV words:")
print("=" * 65)
for word in oov_words:
    similar = ft_model.wv.most_similar(word, topn=3)
    pairs = ', '.join([f"{w}({s:.2f})" for w, s in similar])
    print(f"  {word:<12} -> {pairs}")

FastText - Most Similar In-Vocabulary Words for OOV words:
  smartphones  -> smartphone(1.00), smartwatch(1.00), smart(1.00)
  cameras      -> camera(1.00), condition(1.00), photograph(1.00)
  unfriendly   -> friendly(1.00), crystal(1.00), monitor(1.00)
  chargers     -> charge(1.00), battery(1.00), single(1.00)
  beautifully  -> beautiful(1.00), fantastic(1.00), design(1.00)



Let's check if FastText produces sensible similarity scores for words that *are* in the vocabulary, compared to Word2Vec.

In [6]:
pairs = [
    ('camera',  'photograph'),
    ('battery', 'charge'),
    ('screen',  'display'),
    ('quality', 'premium'),
]

comparison_rows = []
for w1, w2 in pairs:
    w2v_sim = w2v_model.wv.similarity(w1, w2)
    ft_sim = ft_model.wv.similarity(w1, w2)
    comparison_rows.append((w1, w2, round(w2v_sim, 4), round(ft_sim, 4)))

sim_df = pd.DataFrame(comparison_rows, columns=['Word 1', 'Word 2', 'Word2Vec Sim', 'FastText Sim'])
print("Cosine Similarity Comparison for In-Vocabulary Pairs:")
print("=" * 60)
print(sim_df.to_string(index=False))

Cosine Similarity Comparison for In-Vocabulary Pairs:
 Word 1     Word 2  Word2Vec Sim  FastText Sim
 camera photograph        0.9921        0.9994
battery     charge        0.9964        0.9996
 screen    display        0.9695        0.9983
quality    premium        0.9809        0.9983




We will test how well FastText clusters sentiment words compared to Word2Vec.
For this, we'll extract the positive and negative sentiment words present in our vocabulary.

In [7]:
vocab = set(w2v_model.wv.index_to_key)

positive_words = [w for w in
    ['amazing', 'excellent', 'outstanding', 'fantastic',
     'brilliant', 'superb', 'wonderful', 'perfect', 'beautiful', 'satisfied']
    if w in vocab]

negative_words = [w for w in
    ['terrible', 'horrible', 'awful', 'dreadful',
     'disappointed', 'frustrated', 'poor', 'worst', 'ugly', 'sluggish']
    if w in vocab]

print(f"Positive words in vocab: {positive_words}")
print(f"Negative words in vocab: {negative_words}")

Positive words in vocab: ['amazing', 'excellent', 'outstanding', 'fantastic', 'brilliant', 'superb', 'wonderful', 'perfect', 'beautiful', 'satisfied']
Negative words in vocab: ['terrible', 'horrible', 'awful', 'dreadful', 'disappointed', 'frustrated', 'poor', 'worst', 'ugly', 'sluggish']


Now, we calculate the average pairwise similarities within positive words, within negative words, and between positive and negative words.

In [8]:
def calculate_avg_similarities(model):
    # Positive vs Positive
    pp_sims = []
    for i in range(len(positive_words)):
        for j in range(i + 1, len(positive_words)):
            pp_sims.append(model.wv.similarity(positive_words[i], positive_words[j]))
            
    # Negative vs Negative
    nn_sims = []
    for i in range(len(negative_words)):
        for j in range(i + 1, len(negative_words)):
            nn_sims.append(model.wv.similarity(negative_words[i], negative_words[j]))
            
    # Positive vs Negative
    pn_sims = []
    for p_word in positive_words:
        for n_word in negative_words:
            pn_sims.append(model.wv.similarity(p_word, n_word))
            
    return np.mean(pp_sims), np.mean(nn_sims), np.mean(pn_sims)

w2v_pp, w2v_nn, w2v_pn = calculate_avg_similarities(w2v_model)
ft_pp, ft_nn, ft_pn = calculate_avg_similarities(ft_model)

summary_data = {
    'Category': [
        'Positive vs Positive (Same)', 
        'Negative vs Negative (Same)', 
        'Positive vs Negative (Opposite)'
    ],
    'Word2Vec Avg Similarity': [w2v_pp, w2v_nn, w2v_pn],
    'FastText Avg Similarity': [ft_pp, ft_nn, ft_pn]
}

summary_df = pd.DataFrame(summary_data)
print("Sentiment Clustering Performance Comparison:")
print("=" * 65)
print(summary_df.to_string(index=False))

Sentiment Clustering Performance Comparison:
                       Category  Word2Vec Avg Similarity  FastText Avg Similarity
    Positive vs Positive (Same)                 0.969692                 0.997270
    Negative vs Negative (Same)                 0.970734                 0.997038
Positive vs Negative (Opposite)                 0.969724                 0.997135
